
# Reddit insights demo

Two business questions, answered from live Reddit data.

For each question:
1. **Pull the posts** from Reddit (Apify Reddit scraper)
2. **Look at the posts** ourselves, with links to the originals
3. **Ask Claude** to analyse them and sum it up

Runs and Claude answers are cached, so rerunning a cell is instant and free.


In [ ]:

# Setup: keys come from .env and are never printed
from dotenv import load_dotenv
load_dotenv()

import os, getpass
import anthropic
from apify_client import ApifyClient
from IPython.display import Markdown, display
from demo_helpers import fetch_posts, posts_markdown, ClaudeAnalyst

apify = ApifyClient(os.environ.get("APIFY_TOKEN") or getpass.getpass("Apify token: "))
claude = ClaudeAnalyst(anthropic.Anthropic())
print("Ready")



---
## Question 1
### How is everyone's overall sentiment towards Claude AI over the last month?

We search Reddit for "Claude AI", most relevant posts from the past month.


In [ ]:

q1_input = {
    "searchTerms": ["Claude AI"],
    "searchPosts": True,
    "searchComments": False,
    "searchSort": "relevance",
    "searchTime": "month",
    "maxPostsCount": 40,
}
q1 = fetch_posts(apify, q1_input, max_items=40, max_usd=0.25)  # hard caps: 40 results, $0.25


### The posts

In [ ]:

display(Markdown(posts_markdown(q1, n=10)))



### Claude's analysis

Claude reads every post and labels its sentiment towards Claude AI. Posts that only mention it in passing are marked off topic and left out. Then it sums up the main themes.


In [ ]:

q1["sentiment"] = claude.label_sentiment(q1, "Claude AI")
display(Markdown(claude.sentiment_report(q1, "Claude AI")))


Posts again, with Claude's label next to each one:

In [ ]:

display(Markdown(posts_markdown(q1, n=15, label_col="sentiment")))



---
## Question 2
### Which brands are people usually comparing to or mentioning when discussing Nike?

We search Reddit posts and comments for "Nike vs", "Nike alternative" and "Nike compared to", most relevant from the past year.


In [ ]:

q2_input = {
    "searchTerms": ["Nike vs", "Nike alternative", "Nike compared to"],
    "searchPosts": True,
    "searchComments": True,
    "searchSort": "relevance",
    "searchTime": "year",
    "maxPostsCount": 30,     # these limits apply to each search term,
    "maxCommentsCount": 60,  # so 3 terms can return up to 270 rows
}
q2 = fetch_posts(apify, q2_input, max_items=150, max_usd=0.50)  # hard caps: 150 results, $0.50


### The posts

In [ ]:

display(Markdown(posts_markdown(q2, n=10)))



### Claude's analysis

Claude reads every post and comment, pulls out the other sportswear and apparel brands mentioned next to Nike, then sums up how people compare them.


In [ ]:

q2["brands"] = claude.extract_brands(q2, "Nike")
# Jordan and Converse are owned by Nike, so they are not counted as competitors
display(Markdown(claude.brand_report(q2, "Nike", exclude=["Jordan", "Air Jordan", "Converse"])))


Posts again, with the brands Claude found in each one:

In [ ]:

q2["brands_found"] = q2["brands"].apply(lambda l: ", ".join(l) if l else "")
display(Markdown(posts_markdown(q2[q2["brands_found"] != ""], n=15, label_col="brands_found")))



---
## Notes on the data

* Reddit only. Other platforms may tell a different story.
* These are samples (40 posts for Claude AI, 240 posts and comments for Nike), good for a direction, not a precise number. Bigger runs cost more but give a firmer answer.
* Cost of these two runs: about $0.10 for Claude AI and $0.50 for Nike on Apify, plus a few cents of Claude.
* Search picks the most relevant posts for the time window, so very popular threads count more than quiet ones.
* Claude's labels are a judgement call. The post tables let you check any of them.
